In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="FBK-MT/Speech-MASSIVE", 
                  repo_type="dataset", local_dir="./Speech-MASSIVE")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 95 files: 100%|██████████| 95/95 [00:23<00:00,  3.96it/s]


'/home/ubuntu/Speech-MASSIVE'

In [4]:
files = glob('Speech-MASSIVE/*/*.parquet')
files = [f for f in files if 'all/' in f]
files

['Speech-MASSIVE/all/train-00010-of-00018.parquet',
 'Speech-MASSIVE/all/validation-00008-of-00019.parquet',
 'Speech-MASSIVE/all/train-00004-of-00018.parquet',
 'Speech-MASSIVE/all/validation-00005-of-00019.parquet',
 'Speech-MASSIVE/all/train-00017-of-00018.parquet',
 'Speech-MASSIVE/all/train-00007-of-00018.parquet',
 'Speech-MASSIVE/all/train-00012-of-00018.parquet',
 'Speech-MASSIVE/all/train-00013-of-00018.parquet',
 'Speech-MASSIVE/all/validation-00014-of-00019.parquet',
 'Speech-MASSIVE/all/train-00014-of-00018.parquet',
 'Speech-MASSIVE/all/validation-00018-of-00019.parquet',
 'Speech-MASSIVE/all/validation-00009-of-00019.parquet',
 'Speech-MASSIVE/all/train-00006-of-00018.parquet',
 'Speech-MASSIVE/all/validation-00017-of-00019.parquet',
 'Speech-MASSIVE/all/validation-00006-of-00019.parquet',
 'Speech-MASSIVE/all/train-00000-of-00018.parquet',
 'Speech-MASSIVE/all/train_115-00000-of-00002.parquet',
 'Speech-MASSIVE/all/train-00001-of-00018.parquet',
 'Speech-MASSIVE/all/vali

In [7]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['utt'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['speaker_id'].iloc[i]}"
            })
        
    return data

In [6]:
data = loop((files[:1], 0))
data

  0%|          | 0/1279 [00:00<?, ?it/s]


[{'audio_filename': 'Speech-MASSIVE_audio/Speech-MASSIVE-all-train-00010-of-00018_0.mp3',
  'text': "quelle proportion de la terre est constituée d'eau",
  'speaker': 'Speech-MASSIVE_audio_6598afcfcecc69fbfd0943cc'}]

In [8]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 1284/1284 [01:28<00:00, 14.57it/s]


In [9]:
len(data)

48796

In [10]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'Speech-MASSIVE_audio/Speech-MASSIVE-all-train-00010-of-00018_0.mp3',
 'text': "quelle proportion de la terre est constituée d'eau",
 'speaker': 'Speech-MASSIVE_audio_6598afcfcecc69fbfd0943cc'}

In [11]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'Speech-MASSIVE')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 63.09ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  68%|██████▊   | 1.26MB / 1.86MB,  150kB/s  
Processing Files (1 / 1): 100%|██████████| 1.86MB / 1.86MB,  218kB/s  
Processing Files (1 / 1): 100%|██████████| 1.86MB / 1.86MB,  217kB/s  
New Data Upload: 100%|██████████| 1.86MB / 1.86MB,  217kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:08<00:00,  8.94s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/c3046e135f11935c46edc89db6600a016e8a39aa', commit_message='Upload dataset', commit_description='', oid='c3046e135f11935c46edc89db6600a016e8a39aa', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [12]:
audio_files = [d['audio_filename'] for d in data]

with open('Speech-MASSIVE-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [13]:
!zip -rq Speech-MASSIVE_audio.zip Speech-MASSIVE_audio

The history saving thread hit an unexpected error (OperationalError('database is locked')).History will not be written to the database.


In [15]:
# !hf upload malaysia-ai/Multilingual-TTS Speech-MASSIVE_audio.zip --repo-type=dataset

In [19]:
# !zip -rq Speech-MASSIVE_audio_neucodec.zip Speech-MASSIVE_audio_neucodec

In [20]:
# !hf upload malaysia-ai/Multilingual-TTS Speech-MASSIVE_audio_neucodec.zip --repo-type=dataset